# Neural Network MNIST Simple Example

From [YouTube - Pytorch Neural Network Example](https://www.youtube.com/watch?v=Jy4wM2X21u0&list=PLhhyoLH6IjfxeoooqP9rhU3HJIAVAJ3Vz&index=3)

In [2]:
# Imports
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision.datasets as datasets
import torchvision.transforms as transforms


# Create Fully Connected Network
class NN(nn.Module):
    def __init__(self, input_size, num_classes): # (28x28 images = 724)
        super(NN, self).__init__() # Calls the initialization method of nn.Module
        self.fc1 = nn.Linear(input_size, 50)
        self.fc2 = nn.Linear(50, num_classes)
        
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

        
# Set Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Hyperparameters
input_size = 784
num_classes = 10
learning_rate = 0.001
batch_size = 64
num_epochs = 5

# Load Data
train_dataset = datasets.MNIST(
    root='dataset/',                    # dataset for saving the data
    train=True,                         # Use the training set?
    transform=transforms.ToTensor(),    # Transform the data to tensors
    download=True                       # download if not present in the `dataset/` directory
    )

test_dataset = datasets.MNIST(
    root='dataset/',                    # dataset for saving the data
    train=False,                         # Use the training set?
    transform=transforms.ToTensor(),    # Transform the data to tensors
    download=True                       # download if not present in the `dataset/` directory
    )


train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=True)

# Initialize Network
model = NN(input_size=input_size, num_classes=num_classes).to(device)

# Loss and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Check accuracy on training and test
def check_accuracy(loader, model, epoch):
    if loader.dataset.train:
        print("Testing accuracy on training data")
    
    num_correct = 0
    num_samples = 0
    model.eval()
    
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device=device)
            y = y.to(device=device)
            
            x = x.reshape(x.shape[0], -1)
            
            scores = model(x)
            
            # 64x10
            _, predictions = scores.max(1) # index of max value
            num_correct += (predictions == y).sum()   # Number of correct predictions
            num_samples += predictions.size(0)        # Number of samples
            
        print(f"Epoch {epoch+1}: {num_correct:,} / {num_samples:,} | Accuracy {float(num_correct)/float(num_samples)*100:.2f}")
            
            

# Train Network
for epoch in range(num_epochs):
    for batch_idx, (data, targets) in enumerate(train_loader):
        data = data.to(device=device)
        targets = targets.to(device=device)
        
        # Get to the right input shape
        data = data.reshape(data.shape[0], -1) # flatten inputs into single dimension
        
        # Forward
        scores = model(data)
        loss = criterion(scores, targets)
        
        # Backward
        optimizer.zero_grad() # Set all the gradients to 0 for each batch
        loss.backward()
        
        # Gradient descent or adam step
        optimizer.step()
         
    check_accuracy(test_loader, model, epoch)


KeyboardInterrupt: 